# Caught & Shared — Contrastive Alignment Approach

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
torch.manual_seed(42)

#### 1. Synthetic Data

In [ ]:
num_users = 25
num_items = 80

# User-item interactions
interactions = []
for u in range(num_users):
    num_inter = np.random.randint(4, 12)
    items = np.random.choice(num_items, num_inter, replace=False)
    for i in items:
        interactions.append((u, i))

inter_df = pd.DataFrame(interactions, columns=['user_id', 'item_id'])

# Mentor assignments
multi_mentor_edges = [
    (0, 5, 0.75),
    (0, 7, 0.35),
    (1, 12, 0.85),
    (2, 5, 0.65),
    (3, 8, 0.90),
    (4, 15, 0.60),
]

#### 2. Contrastive Loss + Model

In [ ]:
class ContrastiveRecModel(torch.nn.Module):
    def __init__(self, emb_dim=64):
        super().__init__()
        self.user_encoder = torch.nn.Linear(16, emb_dim)
        self.item_encoder = torch.nn.Linear(16, emb_dim)
        
    def forward(self, user_features, item_features):
        user_emb = self.user_encoder(user_features)
        item_emb = self.item_encoder(item_features)
        return F.normalize(user_emb, p=2, dim=1), F.normalize(item_emb, p=2, dim=1)


def contrastive_loss(user_emb, mentor_emb, temperature=0.07):
    """InfoNCE-style loss to pull user and mentor closer"""
    similarity = torch.matmul(user_emb, mentor_emb.T) / temperature
    labels = torch.arange(user_emb.size(0), device=user_emb.device)
    return F.cross_entropy(similarity, labels)


model = ContrastiveRecModel()

#### 3. Recommendation Function with Contrastive Influence

In [ ]:
@torch.no_grad()
def get_recommendations_contrastive(model, user_features, item_features, 
                                   user_id, mentor_config=None, k=8):
    """
    mentor_config: list of (mentor_id, alpha)
    """
    model.eval()
    
    user_embs, item_embs = model(user_features, item_features)
    target_user_emb = user_embs[user_id].unsqueeze(0)
    
    if mentor_config and len(mentor_config) > 0:
        final_emb = target_user_emb.clone()
        
        for mentor_id, alpha in mentor_config:
            mentor_emb = user_embs[mentor_id].unsqueeze(0)
            final_emb += alpha * (mentor_emb - target_user_emb)
        
        final_emb = F.normalize(final_emb, p=2, dim=1)
    else:
        final_emb = target_user_emb
    
    scores = torch.matmul(final_emb, item_embs.T).squeeze(0)
    topk_scores, topk_indices = torch.topk(scores, k=k)
    
    return {
        'user_id': user_id,
        'mentors': mentor_config,
        'recommended_items': topk_indices.tolist(),
        'scores': topk_scores.tolist()
    }


user_features = torch.randn(num_users, 16)
item_features = torch.randn(num_items, 16)

rec_base = get_recommendations_contrastive(model, user_features, item_features, 
                                          user_id=0, mentor_config=None)
print(f"User 0 (no mentor)     → {rec_base['recommended_items']}")

rec_single = get_recommendations_contrastive(model, user_features, item_features, 
                                            user_id=0, mentor_config=[(5, 0.8)])
print(f"User 0 + Mentor 5     → {rec_single['recommended_items']}")

rec_multi = get_recommendations_contrastive(model, user_features, item_features, 
                                           user_id=0, mentor_config=[(5, 0.65), (7, 0.35)])
print(f"User 0 + 2 Mentors    → {rec_multi['recommended_items']}")